# Consolidação dos resultados dos classificadores meta

Este notebook lê os CSVs gerados por `scripts/train_meta_classifiers.py` e `scripts/train_meta_transfer.py` em `data/meta_results/` e consolida os resultados em uma tabela e um conjunto de figuras.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path('..').resolve()
RESULTS_DIR = REPO_ROOT / 'data' / 'meta_results'
FIG_DIR = REPO_ROOT / 'docs' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print('Results dir:', RESULTS_DIR)
print('Figures dir:', FIG_DIR)

## 1. Carregamento dos CSVs

In [ ]:
indomain_files = {
    ('diabetes', 'pyhard'): 'meta_cv_diabetes_pyhard.csv',
    ('diabetes', 'hcat'): 'meta_cv_diabetes_hcat.csv',
    ('covertype', 'pyhard'): 'meta_cv_covertype_pyhard.csv',
    ('covertype', 'hcat'): 'meta_cv_covertype_hcat.csv',
}
transfer_files = {
    'pyhard': 'meta_transfer_pyhard.csv',
    'hcat': 'meta_transfer_hcat.csv',
}

indomain_frames = []
for (dataset, tool), fname in indomain_files.items():
    df = pd.read_csv(RESULTS_DIR / fname)
    indomain_frames.append(df)
df_in = pd.concat(indomain_frames, ignore_index=True)

transfer_frames = []
for tool, fname in transfer_files.items():
    df = pd.read_csv(RESULTS_DIR / fname)
    df['setting'] = 'transfer_diabetes_to_covertype'
    transfer_frames.append(df)
df_tr = pd.concat(transfer_frames, ignore_index=True)

print('in-domain shape:', df_in.shape)
print('transfer shape:', df_tr.shape)
df_in.head()

## 2. Tabela consolidada in-domain (4 dataset×tool × 4 modelos)

In [ ]:
keep = ['dataset', 'tool', 'model',
        'accuracy_mean', 'precision_mean', 'recall_mean', 'f1_mean',
        'roc_auc_mean', 'average_precision_mean']

tbl_in = (
    df_in[keep]
    .rename(columns={
        'accuracy_mean': 'accuracy', 'precision_mean': 'precision',
        'recall_mean': 'recall', 'f1_mean': 'f1',
        'roc_auc_mean': 'roc_auc', 'average_precision_mean': 'average_precision',
    })
    .sort_values(['dataset', 'tool', 'roc_auc'], ascending=[True, True, False])
    .reset_index(drop=True)
)
tbl_in = tbl_in.round(4)
tbl_in

In [ ]:
tbl_in.to_csv(
    Path("..") / "data" / "meta_results" / "consolidated_indomain.csv",
    index=False,
)


## 3. Tabela consolidada de transferência

In [ ]:
tbl_tr = (
    df_tr[['tool', 'model', 'accuracy', 'precision', 'recall', 'f1',
           'roc_auc', 'average_precision']]
    .sort_values(['tool', 'roc_auc'], ascending=[True, False])
    .reset_index(drop=True)
    .round(4)
)
tbl_tr.to_csv(TABLES_EXPORT / 'meta_transfer_consolidated.csv', index=False)
tbl_tr

## 4. Figura — ROC-AUC por modelo, trilha e dataset (in-domain)

In [ ]:
pivot_auc = tbl_in.pivot_table(
    index='model', columns=['dataset', 'tool'], values='roc_auc',
).loc[['Random Forest', 'Gradient Boosting', 'Logistic Regression', 'XGBoost']]
pivot_auc.columns = [f'{d}-{t}' for d, t in pivot_auc.columns]

fig, ax = plt.subplots(figsize=(9, 4.5))
pivot_auc.plot(kind='bar', ax=ax, width=0.78, colormap='viridis')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='Acaso (0,5)')
ax.set_ylim(0.0, 1.0)
ax.set_ylabel('ROC-AUC médio (5-fold CV)')
ax.set_xlabel('Modelo')
ax.set_title('ROC-AUC dos classificadores meta por dataset e trilha (in-domain)')
ax.legend(title='Dataset · Trilha', loc='upper right', fontsize=9)
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_meta_indomain_roc_auc.png', dpi=160)
plt.show()

## 5. Figura — AP (PRC-AUC) por modelo, trilha e dataset (in-domain)

In [ ]:
pivot_ap = tbl_in.pivot_table(
    index='model', columns=['dataset', 'tool'], values='average_precision',
).loc[['Random Forest', 'Gradient Boosting', 'Logistic Regression', 'XGBoost']]
pivot_ap.columns = [f'{d}-{t}' for d, t in pivot_ap.columns]

fig, ax = plt.subplots(figsize=(9, 4.5))
pivot_ap.plot(kind='bar', ax=ax, width=0.78, colormap='plasma')
ax.axhline(0.375, color='gray', linestyle='--', linewidth=1,
           label='Prevalência (0,375)')
ax.set_ylim(0.0, 1.0)
ax.set_ylabel('Average precision (PRC-AUC) médio (5-fold CV)')
ax.set_xlabel('Modelo')
ax.set_title('PRC-AUC dos classificadores meta por dataset e trilha (in-domain)')
ax.legend(title='Dataset · Trilha', loc='upper right', fontsize=9)
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_meta_indomain_average_precision.png', dpi=160)
plt.show()

## 6. Figura — Transferência Diabetes → Covertype

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, metric, title in zip(
    axes, ['roc_auc', 'average_precision'], ['ROC-AUC', 'Average precision (PRC-AUC)']
):
    pivot = tbl_tr.pivot_table(index='model', columns='tool', values=metric).loc[
        ['Random Forest', 'Gradient Boosting', 'Logistic Regression', 'XGBoost']
    ]
    pivot.plot(kind='bar', ax=ax, width=0.7, color=['#2a9d8f', '#e76f51'])
    reference = 0.5 if metric == 'roc_auc' else 0.375
    ax.axhline(reference, color='gray', linestyle='--', linewidth=1,
               label=f'Referência ({reference})')
    ax.set_ylim(0.0, 1.0)
    ax.set_title(f'{title} — Treino Diabetes → Teste Covertype')
    ax.set_xlabel('Modelo')
    ax.tick_params(axis='x', rotation=20)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(title='Trilha', loc='upper right', fontsize=9)
axes[0].set_ylabel('Métrica (teste)')
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_meta_transfer.png', dpi=160)
plt.show()

## 7. Figura — In-domain vs. transferência (só PyHard, melhor modelo)

In [ ]:
summary = pd.DataFrame({
    'Cenário': [
        'Diabetes in-domain\n(PyHard)',
        'Covertype in-domain\n(PyHard)',
        'Diabetes → Covertype\n(PyHard, transfer)',
    ],
    'ROC-AUC': [
        tbl_in.query("dataset=='diabetes' and tool=='pyhard'")['roc_auc'].max(),
        tbl_in.query("dataset=='covertype' and tool=='pyhard'")['roc_auc'].max(),
        tbl_tr.query("tool=='pyhard'")['roc_auc'].max(),
    ],
    'PRC-AUC': [
        tbl_in.query("dataset=='diabetes' and tool=='pyhard'")['average_precision'].max(),
        tbl_in.query("dataset=='covertype' and tool=='pyhard'")['average_precision'].max(),
        tbl_tr.query("tool=='pyhard'")['average_precision'].max(),
    ],
})
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(summary))
width = 0.35
bars1 = ax.bar(x - width/2, summary['ROC-AUC'], width, color='#1f77b4', label='ROC-AUC')
bars2 = ax.bar(x + width/2, summary['PRC-AUC'], width, color='#ff7f0e', label='PRC-AUC')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(summary['Cenário'], fontsize=9)
ax.set_ylim(0.0, 1.0)
ax.set_ylabel('Valor da métrica')
ax.set_title('Melhor classificador meta — trilha PyHard')
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', alpha=0.3)
for b in list(bars1) + list(bars2):
    ax.annotate(f"{b.get_height():.3f}", xy=(b.get_x() + b.get_width()/2, b.get_height()),
                xytext=(0, 2), textcoords='offset points', ha='center', fontsize=8)
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig_meta_pyhard_generalizacao.png', dpi=160)
plt.show()

## 8. Geração da tabela Top 20 em Markdown (substitui a antiga)

In [ ]:
lines = []
lines.append('# Resultados consolidados — classificadores meta (PyHard vs H-CAT)')
lines.append('')
lines.append('Fonte oficial: `data/meta_results/meta_cv_*.csv` e `data/meta_results/meta_transfer_*.csv`.  ')
lines.append('Protocolo: 5-fold stratificado (in-domain); treino completo Diabetes + teste único Covertype (transferência).  ')
lines.append('Cenário de perturbação: 10+20+30% combinados (prevalência 0,375).  ')
lines.append('Todas as métricas são arredondadas a 4 dígitos.')
lines.append('')
lines.append('## In-domain (validação cruzada 5-fold)')
lines.append('')
lines.append('| Dataset | Trilha | Modelo | Accuracy | Precision | Recall | F1 | ROC-AUC | Avg. precision |')
lines.append('|---------|--------|--------|---------:|----------:|-------:|----:|--------:|---------------:|')
for _, r in tbl_in.iterrows():
    lines.append(
        f"| {r['dataset']} | {r['tool']} | {r['model']} | "
        f"{r['accuracy']:.4f} | {r['precision']:.4f} | {r['recall']:.4f} | "
        f"{r['f1']:.4f} | **{r['roc_auc']:.4f}** | {r['average_precision']:.4f} |"
    )
lines.append('')
lines.append('## Transferência — treino Diabetes, teste Covertype')
lines.append('')
lines.append('| Trilha | Modelo | Accuracy | Precision | Recall | F1 | ROC-AUC | Avg. precision |')
lines.append('|--------|--------|---------:|----------:|-------:|----:|--------:|---------------:|')
for _, r in tbl_tr.iterrows():
    lines.append(
        f"| {r['tool']} | {r['model']} | "
        f"{r['accuracy']:.4f} | {r['precision']:.4f} | {r['recall']:.4f} | "
        f"{r['f1']:.4f} | **{r['roc_auc']:.4f}** | {r['average_precision']:.4f} |"
    )

md_content = '\n'.join(lines) + '\n'
(REPO_ROOT / 'docs' / 'tabela_top20_resultados_classificadores.md').write_text(md_content, encoding='utf-8')
print('Arquivo reescrito:', REPO_ROOT / 'docs' / 'tabela_top20_resultados_classificadores.md')
print(md_content)